## Imports

In [57]:
# general
import random
import numpy as np
import pandas as pd
from collections import defaultdict
import random
import json

## Data handling


In [63]:
def tokenize(txtFile):
    stop_sentence_tokens = [".", "!", "?"]
    interrupt_sentence_tokens = [",", ";", ":"]

    words = ["<s>"]
    word = []
    for character in txtFile.read().lower():
        if character == "'" or character == '"' or character=='`':
            continue
        if character == "\n":
            if word:
                words.append("".join(word))
                words.append("</s>")
                words.append("<s>")            
                word = []
            continue    
        if character == " ":
            if word:
                words.append("".join(word))
                word = []
            continue
        if character in stop_sentence_tokens:
            if word:
                words.append("".join(word))
                words.append(character)
                words.append("</s>")
                words.append("<s>")
                word = []
            continue
        if character in interrupt_sentence_tokens:
            if word:
                words.append("".join(word))
                words.append(character)
                word = []
            continue
        word.append(character)
    return words

def vocabulary(words):
    return list(set(words))

def sentenceinizer(words):
    sentences = []
    sentence = []
    for word in words:
        sentence.append(word)
        if word == "</s>":
            sentences.append(sentence)
            sentence = []
    return sentences

def to_ngram_data(sentences, n=3):
    temp_sentences = [list(sentence) for sentence in sentences]
    for sentence in temp_sentences:
        for i in range(n):  
            sentence.insert(0, "<s>")
    return temp_sentences

In [64]:
txtFile = open("data/lotr.txt")
words = tokenize(txtFile)
vocab = vocabulary(words)
sentences = sentenceinizer(words)

## NGram

In [60]:
def compute_ngram_frequencies(sentences, n):
    ngram_frequencies = {}

    for sentence in sentences:
        for ngram in zip(*[sentence[i:] for i in range(n)]):
            key = ngram[:-1]     # first n-1 words
            next_word = ngram[-1] # last word

            if key not in ngram_frequencies:
                ngram_frequencies[key] = {}

            ngram_frequencies[key][next_word] = \
                ngram_frequencies[key].get(next_word, 0) + 1

    return ngram_frequencies

def ngram_frequencies_analysis(ngram_frequencies, top_n=10):
    total_ngrams = 0
    unique_ngrams = 0
    flat_ngrams = []

    for key, next_words in ngram_frequencies.items():
        for next_word, count in next_words.items():
            total_ngrams += count
            unique_ngrams += 1
            flat_ngrams.append(((*key, next_word), count))

    sorted_ngrams = sorted(
        flat_ngrams,
        key=lambda x: x[1],
        reverse=True
    )

    print(f"Total n-grams: {total_ngrams}")
    print(f"Unique n-grams: {unique_ngrams}\n")

    print(f"Top {top_n} most frequent n-grams:")
    for ngram, count in sorted_ngrams[:top_n]:
        print(f"{ngram} -> {count}")

def ngram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def ngram_inference(ngram_frequencies, n, current_words=None):
    if current_words is None:
        current_words = ("<s>",) * (n - 1)

    result = list(current_words)

    while True:
        key = tuple(current_words)

        if key not in ngram_frequencies:
            print(key)
            break

        next_words = ngram_frequencies[key]
        next_word = ngram_generate_next_word(next_words)

        result.append(next_word)

        if next_word == "</s>":
            break

        current_words = current_words[1:] + (next_word,)

    return result


def word_list_print(words):
    result = ''
    for word in words:
        if word in ["<s>", "</s>"]:
            continue

        if word in [".", ",", "!", "?", ":", ";"]:
            result = result.rstrip()
            result += word + " "
        else:
            result += word + " "
    result = result.strip()
    if result:
        result = result[0].upper() + result[1:]

    print(result)

def prune_frequencies(ngram_frequencies, threshold=2):
    pruned = {}

    for context, next_words in ngram_frequencies.items():
        filtered = {
            word: count
            for word, count in next_words.items()
            if count >= threshold
        }

        if filtered:
            pruned[context] = filtered

    return pruned

In [65]:
n = 10
ngram_sentences = to_ngram_data(sentences, n)

ngram_frequencies = compute_ngram_frequencies(ngram_sentences, n)
pruned_frequencies = prune_frequencies(ngram_frequencies)

for i in range(10):
    sentence = ngram_inference(ngram_frequencies, n)
    word_list_print(sentence)

The blow caught him on the right side, and frodo was hurled against the wall and pinned.
Why would that make you happy?
Curse it!
Frodo did not want to have any more to do with the party.
That is far south in isengard, in the end of the misty mountains, not far from the gap of rohan.
It still seemed very dark, he thought.
Look, my friends!
At the sign of the prancing pony bree was the chief village of the bree-land, a small inhabited region, like an island in the empty lands round about.
Northward, where they looked most hopefully, they could see nothing that might be the line of the great east road, for which they were making.
What are they?


## Backoff ngram Model

In [ ]:
def backoff_ngram_model(n=5):
    model_dictionary = {}
    for i in range(n, 0, -1):
        ngram_sentences = to_ngram_data(sentences, i)
        ngram_frequencies = compute_ngram_frequencies(ngram_sentences, i)
        pruned_frequencies = prune_frequencies(ngram_frequencies)
        model_dictionary["{0}gram_model".format(i)] = pruned_frequencies
    return model_dictionary

def backoff_ngram_inference(model_dictionary, max_n, current_words=None):
    if current_words is None:
        current_words = ("<s>",) * (max_n - 1)

    result = list(current_words)

    while True:
        next_word = None
        print(max_n, len(model_dictionary))

        for n in range(max_n, 0, -1):
            model = model_dictionary[f"{n}gram_model"]

            context_size = n - 1
            key = tuple(current_words[-context_size:]) if context_size > 0 else ()

            if key in model:
                next_words = model[key]
                next_word = ngram_generate_next_word(next_words)
                break 

        if next_word is None:
            break

        result.append(next_word)

        if next_word == "</s>":
            break

        current_words = current_words + (next_word,)

    return result
    
def ngram_generate_next_word(next_words_dict):
    words = list(next_words_dict.keys())
    weights = list(next_words_dict.values())
    return random.choices(words, weights=weights)[0]

def generate_sentences(models, n, sentences=10):
    for i in range(sentences):
        sentence = backoff_ngram_inference(models, n)
        word_list_print(sentence)

In [ ]:
n = 10

models = backoff_ngram_model(n)
generate_sentences(models, n)

10 10


KeyError: 9

## Save Model

In [ ]:
def save_ngram_model(ngram_frequencies, filename):
    serializable = {}

    for key, next_words in ngram_frequencies.items():
        # Convert tuple key → string
        key_str = " ".join(key)
        serializable[key_str] = next_words

    with open(filename, "w") as f:
        json.dump(serializable, f, indent=4)

def load_ngram_model(filename):
    with open(filename, "r") as f:
        data = json.load(f)

    ngram_frequencies = {}

    for key_str, next_words in data.items():
        key = tuple(key_str.split())
        ngram_frequencies[key] = next_words

    return ngram_frequencies

In [ ]:
save_ngram_model(ngram_frequencies, "model.json")
ngram_frequencies = load_ngram_model("model.json")

## Improvements

Smoothing - Kneser-Ney

Pruning

Backoff models